In [1]:
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
plt.rcParams.update({
    'figure.figsize': (3.6, 2.7), 'font.size': 15, 'lines.linewidth': 2,
    'xtick.labelsize': 'small', 'ytick.labelsize': 'small',
    'legend.fontsize': 'small', 'axes.titlesize': 'medium',
    'axes.spines.top': False, 'axes.spines.right': False,
    'image.interpolation': 'nearest', 'savefig.dpi': 100,
})

import os
import pickle
from pathlib import Path
import networkx as nx
import numpy as np
from jarvis.utils import tqdm

from hexarena import STORE_DIR
FIG_DIR = Path('figures')
os.makedirs(FIG_DIR, exist_ok=True)
rng = np.random.default_rng()

# Compute Bayesian beliefs in the bandit environment

In [2]:
from hexarena.scripts.common import get_block_ids, create_env

subject, kappa, gamma = 'marco', 0.01, 1
block_ids = get_block_ids(subject, kappa, gamma, min_pos_ratio=0, min_gaze_ratio=0)
print(f"{len(block_ids)} blocks found for {subject.capitalize()} (kappa={kappa}, gamma={gamma})")

45 blocks found for Marco (kappa=0.01, gamma=1)


In [3]:
from irc.utils import check_env

env = create_env(gamma, no_arena=True)
check_env(env)

In [ ]:
from hexarena.utils import load_monkey_data, align_monkey_data
import irc

for session_id, block_idx in tqdm(block_ids, unit='block'):
    block_data = load_monkey_data(subject, session_id, block_idx)
    align_monkey_data(block_data)
    env_data = env.convert_experiment_data(block_data)
    _, env_states, obss, actions = env.extract_episode(env_data)
    beliefs, infos = irc.bayesian(
        env, actions, obss,
        ckpt_pth=STORE_DIR/'beliefs'/subject/f'beliefs_{session_id}[block{block_idx}].pkl',
        bmdp_kw={
            'train_kw.n_epochs': 40,
            'estimate_kw.sga_kw.n_epochs': 60,
        },
        pbar_kw={'disable': True},
    )

  0%|                                                                                                         …